In [6]:
import torch
import torch.nn as nn


# Conv + BatchNorm + ReLU6

class ConvBNReLU(nn.Sequential):
    def __init__(self, in_channels, out_channels,
                 kernel_size=3, stride=1, groups=1):

        padding = (kernel_size - 1) // 2

        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                groups=groups,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        )

# Inverted Residual Block

class InvertedResidual(nn.Module):
    def __init__(self, in_channels,
                 out_channels,
                 stride,
                 expand_ratio):

        super().__init__()

        hidden_dim = in_channels * expand_ratio

        self.use_residual = (
            stride == 1 and
            in_channels == out_channels
        )

        layers = []

        # 1x1 expansion
        if expand_ratio != 1:
            layers.append(
                ConvBNReLU(
                    in_channels,
                    hidden_dim,
                    kernel_size=1
                )
            )

        # depthwise
        layers.append(
            ConvBNReLU(
                hidden_dim,
                hidden_dim,
                kernel_size=3,
                stride=stride,
                groups=hidden_dim
            )
        )

        # pointwise projection
        layers.append(
            nn.Conv2d(
                hidden_dim,
                out_channels,
                kernel_size=1,
                bias=False
            )
        )

        layers.append(
            nn.BatchNorm2d(out_channels)
        )

        self.block = nn.Sequential(*layers)

    def forward(self, x):

        if self.use_residual:
            return x + self.block(x)

        return self.block(x)

# MobileNetV2

class MobileNetV2(nn.Module):
    def __init__(self,
                 num_classes=1000):

        super().__init__()

        input_channel = 32
        last_channel = 1280

        # t = expand ratio
        # c = output channels
        # n = repeats
        # s = stride
        settings = [
            [1, 16, 1, 1],
            [6, 24, 2, 2],
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        layers = []

        # first conv
        layers.append(
            ConvBNReLU(
                3,
                input_channel,
                stride=2
            )
        )

        # inverted residual blocks
        for t, c, n, s in settings:

            for i in range(n):

                stride = s if i == 0 else 1

                layers.append(
                    InvertedResidual(
                        input_channel,
                        c,
                        stride,
                        t
                    )
                )

                input_channel = c

        # final conv
        layers.append(
            ConvBNReLU(
                input_channel,
                last_channel,
                kernel_size=1
            )
        )

        self.features = nn.Sequential(*layers)

        # classifier
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            last_channel,
            num_classes
        )

    def forward(self, x):

        x = self.features(x)

        x = self.pool(x)

        x = torch.flatten(x, 1)

        x = self.classifier(x)

        return x



# Test

if __name__ == "__main__":

    model = MobileNetV2(num_classes=1000)

    x = torch.randn(2, 3, 224, 224)

    y = model(x)

    print(y.shape)

torch.Size([2, 1000])
